In [2]:
# Limit threads to avoid potential FAISS/OMP issues
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import huggingface_hub
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download


import yaml
import json         # ← add this
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


/Users/shanzhonghan/Desktop/LLM/project/juris/backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


# Load config
with open("config.yaml", encoding="utf8") as f:
    cfg = yaml.safe_load(f)

# 1) Test embedding
print("Loading embedder...")
embedder = SentenceTransformer(cfg["embed_model"])
vec = embedder.encode(["Hvornår træder bekendtgørelsen i kraft?"], convert_to_numpy=True)
print("Embedding output shape:", vec.shape)

# 2) Test FAISS search
print("Loading FAISS index...")
index = faiss.read_index(cfg["index_path"])
print("Searching index...")
dists, idxs = index.search(vec.astype("float32"), cfg["initial_top_k"])
print("Search distances shape:", dists.shape)
print("Search indices shape:", idxs.shape)
print("First hit metadata index:", idxs[0][0])

# 3) Test chunk extraction
with open(cfg["meta_path"], encoding="utf8") as f:
    meta = json.load(f)
hit = {**meta[idxs[0][0]], "distance": float(dists[0][0])}
from pprint import pprint
print("First hit metadata:")
pprint(hit)
print("Chunk text snippet:")
with open(os.path.join(cfg["out_dir"], f"{hit['law_id']}.json"), encoding="utf8") as f:
    law_doc = json.load(f)
# load_chunk_text logic inline
for chap in law_doc.get("structured_text", []):
    for para in chap.get("paragraphs", []):
        if para["paragraph"] == hit["paragraph"]:
            for sec in para["sections"]:
                if sec["section"] == hit["section"]:
                    text = sec.get("text", "")
                    start = hit.get("char_offset", 0)
                    print(text[start:start + 200])
                    break


Loading embedder...
Embedding output shape: (1, 384)
Loading FAISS index...
Searching index...
Search distances shape: (1, 10)
Search indices shape: (1, 10)
First hit metadata index: 31
First hit metadata:
{'chapter': '',
 'char_offset': 0,
 'distance': 0.2622790038585663,
 'law_id': '2025_933',
 'paragraph': '§ 12.',
 'section': ''}
Chunk text snippet:
Bekendtgørelsen træder i kraft den 1. juli 2025.


In [4]:
from sentence_transformers import CrossEncoder

# 1) Instantiate
cross = CrossEncoder(cfg["cross_encoder_model"])

# 2) Prepare a dummy query + snippet pair
query = "Hvornår træder bekendtgørelsen i kraft?"
snippet = "Bekendtgørelsen træder i kraft den 1. juli 2025."
print("Running rerank…")

# 3) Run and print
scores = cross.predict([[query, snippet]])
print("CrossEncoder score:", scores)


Running rerank…
CrossEncoder score: [6.295732]


In [5]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# 1) Load tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(cfg["gen_model"])
model = AutoModelForSeq2SeqLM.from_pretrained(cfg["gen_model"])
gen = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# 2) Make a minimal prompt
prompt = (
    "You are a Danish legal assistant.\n\n"
    "Context:\nBekendtgørelsen træder i kraft den 1. juli 2025.\n\n"
    "Question: Hvornår træder bekendtgørelsen i kraft?\nAnswer:"
)

# 3) Run and print
out = gen(prompt, max_length=50, do_sample=False, num_beams=4)
print("Generation output:", out)


Device set to use cpu


Generation output: [{'generated_text': '1. juli 2025'}]


## RAG Agent Notebook

Interactive step-by-step testing of `rag_agent.py` components.


In [13]:
# Limit threads to avoid potential FAISS/OMP issues
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import huggingface_hub
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download

import glob
import yaml
import json         # ← add this
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from smolagents import CodeAgent, Tool, InferenceClientModel

In [14]:
# Load config
with open("config.yaml", encoding="utf8") as f:
    cfg = yaml.safe_load(f)
cfg


{'chunk_size': 800,
 'chunk_overlap': 200,
 'embed_model': 'all-MiniLM-L6-v2',
 'gen_model': 'google/flan-t5-small',
 'cross_encoder_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
 'initial_top_k': 10,
 'rerank_top_k': 3,
 'sitemap_root': 'https://www.retsinformation.dk/eli/sitemap.xml',
 'out_dir': 'laws_json',
 'index_path': 'laws_mini.faiss',
 'meta_path': 'laws_mini_meta.json',
 'host': '0.0.0.0',
 'port': 8000,
 'log_level': 'INFO',
 'log_format': '%(asctime)s %(levelname)s %(name)s: %(message)s'}

In [15]:
# Load vector index & metadata
INDEX = faiss.read_index(cfg["index_path"])
with open(cfg["meta_path"], encoding="utf8") as f:
    META = json.load(f)
len(META)


353

In [16]:
# Load full law JSONs
LAW_DOCS = {}
for path in glob.glob(os.path.join(cfg["out_dir"], "*.json")):
    with open(path, encoding="utf8") as f:
        doc = json.load(f)
    LAW_DOCS[doc["id"]] = doc
len(LAW_DOCS)


10

In [17]:
# Define load_chunk_text
def load_chunk_text(hit: dict) -> str:
    law = LAW_DOCS.get(hit["law_id"], {})
    for chap in law.get("structured_text", []):
        for para in chap.get("paragraphs", []):
            if para["paragraph"] == hit["paragraph"]:
                for sec in para["sections"]:
                    if sec["section"] == hit["section"]:
                        text = sec.get("text", "")
                        start = hit.get("char_offset", 0)
                        return text[start : start + cfg["chunk_size"]]
    return ""

# Example:
# load_chunk_text(META[0])


# Example query
load_chunk_text(META[0])

'Denne bekendtgørelse fastsætter rammerne og betingelserne for indførelse og anvendelse af intelligente transportsystemer (ITS) i Danmark.'

In [18]:
# Load models
EMBEDDER      = SentenceTransformer(cfg["embed_model"])
CROSS_ENCODER = CrossEncoder(cfg["cross_encoder_model"])
TOKENIZER     = AutoTokenizer.from_pretrained(cfg["gen_model"])
GEN_MODEL     = AutoModelForSeq2SeqLM.from_pretrained(cfg["gen_model"])
GEN_PIPE      = pipeline("text2text-generation", model=GEN_MODEL, tokenizer=TOKENIZER)
EMBEDDER, CROSS_ENCODER, GEN_PIPE


Device set to use cpu


(SentenceTransformer(
   (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
   (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
   (2): Normalize()
 ),
 <transformers.pipelines.text2text_generation.Text2TextGenerationPipeline at 0x123b29690>)

In [19]:
class RAGTool(Tool):
    name        = "rag_tool"
    description = "Retrieve and answer questions about Danish law using RAG."
    inputs = {
        "query": {
            "type":        "string",
            "description": "The user's legal question",
            "required":    True,
        },
    }
    output_type = "object"

    def forward(self, query: str):
        # 永远使用 config 中的默认值
        top_k    = cfg["initial_top_k"]
        rerank_k = cfg["rerank_top_k"]

        # 1) Retrieve via FAISS + embedder
        vec        = np.array(EMBEDDER.encode([query]), dtype="float32")
        dists, idxs = INDEX.search(vec, top_k)
        hits       = [{**META[i], "distance": float(d)} for d, i in zip(dists[0], idxs[0])]

        # 2) Rerank via CrossEncoder
        texts   = [load_chunk_text(h) for h in hits]
        scores  = CROSS_ENCODER.predict([[query, t] for t in texts])
        top_hits = [
            h for h, _ in sorted(zip(hits, scores),
                                 key=lambda x: x[1], reverse=True)[:rerank_k]
        ]

        # 3) Generate answer with citations in prompt
        context = "\n\n".join(
            f"[Lov {h['law_id']} §{h['paragraph']} stk.{h['section']}] {load_chunk_text(h)}"
            for h in top_hits
        )
        prompt = (
            "You are a Danish legal assistant. Answer concisely using the excerpts below and cite.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        )
        out = GEN_PIPE(
            prompt,
            max_length=512,
            do_sample=False,
            num_beams=4
        )[0]["generated_text"].strip()

        return {"answer": out, "citations": top_hits}
rag_tool = RAGTool()

In [23]:
result = rag_tool.forward("Hvordan skal HR håndtere en medarbejder, der er blevet pålagt elektronisk kontrol som del af et opholdsforbud, så vi sikrer, at både virksomheden og medarbejderen overholder alle relevante krav og procedurer?")
print(result)


Token indices sequence length is longer than the specified maximum sequence length for this model (574 > 512). Running this sequence through the model will result in indexing errors


{'answer': '[Lov 2025_914  8. stk.Stk. 2.] Hvis oplysningerne skal bruges i forbindelse med efterforskning og strafforflgning af overtrdelse af et opholdsforbud, pbud om elektronisk kontrol eller vrige overtrdelser af bekendtgrelsens  2, kan oplysningerne opbevares indtil sagens behandling ved domstolene er endeligt afsluttet. [Lov 2025_925  2. stk.] Vejdirektoratet kan i medfr af denne bekendtgrelse, og i henhold til frdselslovens  92 h, stk. 1, meddele tilladelse til forsg med selvkrende motorkretjer. Vejdirektoratet kan i den forbindelse sikre, and vejmyndigheden, politiet og eventuelle private vejejere er blevet hrt. [Lov 2025_914  3. stk.] Aftagning af det elektroniske udstyr skal foretages af politiet eller kriminalforsorgen efter aftale med politiet.', 'citations': [{'law_id': '2025_914', 'chapter': 'Kapitel 4', 'paragraph': '§ 8.', 'section': 'Stk. 2.', 'char_offset': 0, 'distance': 0.7894816398620605}, {'law_id': '2025_925', 'chapter': '', 'paragraph': '§ 2.', 'section': '', '

In [22]:
model     = InferenceClientModel()
rag_agent = CodeAgent(
    tools=[rag_tool], 
    model=model,
    planning_interval=3)

# 只传 query
resp = rag_agent.run("Hvornår træder bekendtgørelsen i kraft?")

print(resp)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Hvornår træder bekendtgørelsen i kraft?                                                                         │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts survey

### 1.1. Facts given in the task
- The task asks for the date when a specific regulation (bekendtgørelse) comes into force.

### 1.2. Facts to look up
- The specific name or identifier of the regulation (bekendtgørelse) in question.
  - Source: The task does not provide this information, so it would need to be provided or clarified.
- The official publication date of the regulation.
  - Source: The Danish Legal Database (retsinformation.dk) or the Danish Ministry of Justice.
- The specific clause or section of the regulation that states the date of entry into force.
  - Source: The Danish Legal Database (retsinformation.dk) or the official gazette (Lovtidend) where the regulation
was published.

### 1.3. Facts to derive
- The exact date when the regulation comes into force based on the information found in the official publication.
  - Reasoning: This requires extracting the relevant information from the regulation text and interpreting it 
correctly.

## 2. Plan
1. Identify the specific name or identifier of the regulation (bekendtgørelse) in question.
2. Look up the official publication date of the regulation on the Danish Legal Database (retsinformation.dk) or the
Danish Ministry of Justice.
3. Retrieve the full text of the regulation from the Danish Legal Database (retsinformation.dk) or the official 
gazette (Lovtidend).
4. Locate the clause or section of the regulation that specifies the date of entry into force.
5. Extract and interpret the exact date when the regulation comes into force.
6. final_answer(answer)


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  regulation_name = "Lov om sundhedsvæsenets digitale infrastruktur"                                               
  result = rag_tool(query=f"Hvornår træder {regulation_name} i kraft?")                                            
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'answer': '[Lov 2025_853  3. stk.] Bekendtgrelsen trder i kraft den 1. juli 2025. [Lov 2025_925  14. stk.] 
Bekendtgrelsen trder i kraft den 1. juli 2025.', 'citations': [{'law_id': '2025_853', 'chapter': '', 'paragraph': 
'§ 3.', 'section': '', 'char_offset': 0, 'distance': 0.5172918438911438}, {'law_id': '2025_933', 'chapter': '', 
'paragraph': '§ 12.', 'section': '', 'char_offset': 0, 'distance': 0.5172919034957886}, {'law_id': '2025_925', 
'chapter': '', 'paragraph': '§ 14.', 'section': '', 'char_offset': 0, 'distance': 0.5172919034957886}]}

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("1. juli 2025")                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 1. juli 2025

[Step 2: Duration 1.63 seconds| Input tokens: 5,286 | Output tokens: 225]

1. juli 2025
